# NDT cells: checks (coverage + pictures)

How well do the 2000 NDT cells of a variant describe its scans? For every scan: the share of the points (after crop
and ground removal) within 0.8 m of a cell (`cov_0.8`), the coverage of the dense 1 m squares (`dense_cov`), coverage
by range, ... (definitions in `src/lpr/ndt_check.py`), and pictures of 10 random scans: before ground removal, after
ground removal, cells.

- **New variants** (built after the checks became part of `extract_ndt.py`): everything is already on Drive
  (`stats_<seq>.csv`, `previews_<seq>/`); this notebook only shows it (sections 1, 2, 6).
- **Variants built before** (e.g. `2026-10-04_baseline_717e81`): the checks run on the Azure VM.
  Section 3 puts cells that were built in Colab (sequence 00) into the blob container, section 4 prints the command
  for the VM, section 5 copies the results to `<variant>/coverage/` on Drive, section 6 shows them.

No GPU needed. SAS URL in Colab Secrets as `AZURE_SAS`.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
NDT_TAG = ""
NDT_PARAMS = dict()                 # the variant, as in notebook 01 (empty = the default variant)
SEQS = ["00", "02", "05", "06", "08"]
EVERY = 1                           # VM checks: every n-th scan (1 = all; the pictures are always made)
SHOW_PREVIEWS = SEQS                # sequences whose 10 pictures are shown in section 6

## 2. Drive, code, variant

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, subprocess, importlib

url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex, lpr.ndt_check as nc
importlib.reload(ex); importlib.reload(nc)

VARIANT, NDT_FULL, ndt_exists = ex.resolve_ndt_variant(DRIVE, NDT_PARAMS, NDT_TAG)
VDIR = f"{DRIVE}/ndt/{VARIANT}"
assert ndt_exists, f"no variant with these parameters on Drive ({VARIANT})"
print("variant:", VARIANT)

def builtin(seq):
    """True if the stats file of the sequence already has the coverage columns."""
    p = f"{VDIR}/stats_{seq}.csv"
    return os.path.exists(p) and "cov_0.8" in open(p).readline()

BUILTIN = {s: builtin(s) for s in SEQS}
NEED_VM = [s for s in SEQS if not BUILTIN[s]]
print("built-in checks:", [s for s in SEQS if BUILTIN[s]] or "none")
print("checks on the VM:", NEED_VM or "none")

## 3. Cells built in Colab -> blob container (so the VM can read them)
Only for sequences without built-in checks whose cells are on Drive but not in the blob container
(normally 00, about 1 GB, a few minutes). Nothing is uploaded twice.

In [ ]:
for s in NEED_VM:
    tar = f"{VDIR}/cells_{s}.tar"
    if not os.path.exists(tar):
        print(f"{s}: no cells_{s}.tar on Drive (built on Azure only?) - the VM uses its own copy")
        continue
    res = subprocess.run(["python", "scripts/upload_to_azure.py", "--sas_url", userdata.get("AZURE_SAS"),
                          "--src", tar, "--dest", f"ndt/{VARIANT}/cells_{s}.tar"],
                         capture_output=True, text=True)
    print(res.stdout[-500:], res.stderr[-1500:])

## 4. On the Azure VM
Start the VM, `ssh` into it, then in the repository folder (inside `tmux`, so it keeps running if SSH disconnects).
About 0.3 s per scan on 4 cores: all ~17,000 scans in roughly 20-30 minutes.

In [ ]:
if NEED_VM:
    seqs = " ".join(NEED_VM)
    print("tmux new -s checks")
    print(f'git pull && VARIANT={VARIANT} SEQS="{seqs}" EVERY={EVERY} bash scripts/azure_coverage.sh "<SAS URL>"')
    print("\nWhen it says done: stop the VM in the Azure portal, then run section 5.")
else:
    print("nothing to run on the VM: all sequences have built-in checks")

## 5. Results of the VM -> Drive (`<variant>/coverage/`)

In [ ]:
if NEED_VM:
    res = subprocess.run(["python", "scripts/fetch_from_azure.py", "--sas_url", userdata.get("AZURE_SAS"),
                          "--method", "ndt", "--variant", VARIANT, "--seqs", *NEED_VM, "--drive", DRIVE,
                          "--coverage"], capture_output=True, text=True)
    print(res.stdout, res.stderr[-1500:])
else:
    print("nothing to copy")

## 6. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

def table(seq):
    """Per-scan checks of a sequence: built-in (stats file) or from the VM (coverage/)."""
    if BUILTIN.get(seq):
        return pd.read_csv(f"{VDIR}/stats_{seq}.csv").sort_values("idx"), f"{VDIR}/previews_{seq}"
    p = f"{VDIR}/coverage/coverage_{seq}.csv"
    if os.path.exists(p):
        return pd.read_csv(p), f"{VDIR}/coverage/previews_{seq}"
    return None, None

tables, pics = {}, {}
for s in SEQS:
    t, d = table(s)
    if t is None:
        print(f"{s}: no checks yet")
        continue
    tables[s], pics[s] = t, d

if tables:
    summary = pd.DataFrame({s: nc.summarize(t) for s, t in tables.items()}).T
    summary.index.name = "seq"
    with pd.option_context("display.max_columns", None, "display.width", 250):
        display(summary)
    summary.to_csv(f"{VDIR}/checks_summary.csv")

    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    for s, t in tables.items():
        ax[0].hist(t["cov_0.8"], bins=50, histtype="step", label=f"KITTI {s}")
        ax[1].plot(t["idx"], t["cov_0.8"].rolling(25, min_periods=1).mean(), lw=1, label=f"KITTI {s}")
    ax[0].set_xlabel("points within 0.8 m of a cell (%)"); ax[0].set_ylabel("scans"); ax[0].legend()
    ax[1].set_xlabel("scan"); ax[1].set_ylabel("cov_0.8 (%), running mean of 25"); ax[1].legend()
    rng_cols = [c for c in nc.COVERAGE_FIELDS if c.startswith("cov_") and ("-" in c or c.endswith("+"))]
    pd.DataFrame({s: t[rng_cols].mean() for s, t in tables.items()}).plot.bar(ax=ax[2])
    ax[2].set_ylabel("cov_0.8 (%)"); ax[2].set_title("coverage by range from the sensor")
    fig.suptitle(VARIANT); fig.tight_layout()
    fig.savefig(f"{VDIR}/checks_coverage.png", dpi=110)
    plt.show()

In [ ]:
# pictures: before ground removal | after ground removal | cells
for s in SHOW_PREVIEWS:
    for png in sorted(glob.glob(f"{pics.get(s) or '/nonexistent'}/*.png")):
        print(f"KITTI {s}: {os.path.basename(png)}")
        display(Image(png, width=1400))